# BMLL | Corrected four-year portfolio and order book (research replay)

**Analysis window:** 2022-07-01 to 2026-06-30. **Bootstrap:** June 2022 rebalance.

This revision fixes the earlier four-year *global exclusion* bug: a single unresolved event or temporary price issue must not exclude a security for the entire history. It also uses WRDS raw USD dividends as a **provisional research-only convention** when the Bloomberg discrepancy exactly matches a later CRSP-verified split factor or when source economics are otherwise identical.

**Supervisor-handoff changes (Nov 2026)**
* **Rebalance timing:** quarterly orders are now decided at the close of the session before the S&P reference Friday and filled in that **Friday's closing auction**, the print the index itself rebalances at. Before this change, decisions were made on the Monday snapshot and filled at Tuesday's close, 2–3 sessions late. `EXECUTION_TIMING='next_session'` reproduces the old run exactly.
* **Benchmark:** daily S&P 500 total return (CRSP `dsp500` `vwretd`) is extracted from WRDS, with tracking difference and tracking error reported over the evaluation window.
* **Investor flows:** an optional seeded synthetic subscription/redemption scenario, dealt at the closing auction with one-session notice.
* Code clean-up: configuration validation, evaluation-period return base, duplicate imports, shadowed variables.

**Certification is NOT automatic.** Unapproved events, ticker-only matches, complex actions, vendor data gaps, and missing identifier evidence remain flagged. Real WRDS access is required; the result must be independently validated after the live run.

No Python source or vendor data is embedded in this notebook.

## 0. Configuration

Change dates and source release here, before running any subsequent cell. Defaults represent four complete analysis years.

In [ ]:
ANALYSIS_START = '2022-07-01'
END_DATE = '2026-06-30'
REBALANCE_START = '2022-01-01'  # discover the June-2022 starting holdings
CRSP_SCHEMA = 'crsp_q_stock'   # try crsp_a_stock if the quarterly release lacks history
AUM = 10_000_000_000.0
CASH_BUFFER_BPS = 10.0
FEE_BPS = 0.0

# Rebalance execution. 'reference_close' = closing auction of the S&P reference
# Friday (decided the session before). 'next_session' = old Monday/Tuesday run.
EXECUTION_TIMING = 'reference_close'

# Synthetic investor flows (set USE_SYNTHETIC_FLOWS=False for the no-flow baseline).
USE_SYNTHETIC_FLOWS = True
FLOW_SEED = 7
FLOW_DAILY_SIGMA_BPS = 5.0     # retail-like daily noise, bps of NAV
FLOW_LUMPY_PROB = 0.01         # chance per session of an institutional ticket
FLOW_LUMPY_BPS = (50.0, 300.0) # ticket size range, bps of NAV, random sign
FLOW_BAND_BPS = 25.0           # invest excess cash only beyond this band

RUN_OFFLINE_TESTS = True
INSTALL_DEPENDENCIES = True    # set False once the kernel has the requirements
RUN_WRDS_EXTRACTION = True
RUN_BENCHMARK_EXTRACTION = True
RUN_REPLAY = True
# RESEARCH ONLY: deterministic WRDS convention for narrow documented exceptions.
# This does not make corporate actions human-approved or certify full-index coverage.
PROVISIONAL_WRDS_RESEARCH = True

import pandas as pd
_start, _end, _reb = (pd.Timestamp(x) for x in (ANALYSIS_START, END_DATE, REBALANCE_START))
if not _reb < _start < _end:
    raise ValueError('Need REBALANCE_START < ANALYSIS_START < END_DATE')
if EXECUTION_TIMING not in ('reference_close', 'next_session'):
    raise ValueError('EXECUTION_TIMING must be reference_close or next_session')
if not (0 <= CASH_BUFFER_BPS < 10_000 and 0 <= FEE_BPS < 10_000 and AUM > 0):
    raise ValueError('Invalid AUM / cash buffer / fee')
print(f'Analysis window: {ANALYSIS_START} through {END_DATE} | AUM = ${AUM:,.0f} | '
      f'timing = {EXECUTION_TIMING} | synthetic flows = {USE_SYNTHETIC_FLOWS}')

## 1. Source files and Bloomberg upload

When running **from the GitHub checkout root**, the notebook automatically uses repository Python modules and asks only for the Bloomberg workbook.
When running **standalone in Colab**, upload the `BMLL_4Y_Coverage_Fix_Support.zip` and Bloomberg workbook together.
Raw Bloomberg/WRDS data must remain outside version control.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, uuid
from datetime import datetime, timezone

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

# A git checkout already contains the source modules; no source ZIP needed.
_in_repo = all((Path.cwd()/name).is_file() for name in [
    'research_pipeline.py',
    'fund_simulator.py',
    'BMLL_Stage1_2_Corporate_Actions_WRDS_Bloomberg.ipynb',
    'scripts/coverage_recovery.py',
    'scripts/complete_release.py',
    'scripts/build_wrds_handoff.py',
    'scripts/extract_bloomberg.py',
])
if _in_repo:
    PROJECT_DIR = Path.cwd().resolve()
    RUN_ROOT = PROJECT_DIR
    if colab_files is not None:
        print('Repository files detected. Upload Bloomberg Excel workbook only (plus optional review CSVs).')
        _uploads = colab_files.upload()
        workbooks = [Path(k).resolve() for k in _uploads if k.lower().endswith('.xlsx')]
    else:
        workbooks = [Path(input('Bloomberg .xlsx workbook path: ').strip()).expanduser().resolve()]
    if len(workbooks) != 1 or not workbooks[0].is_file():
        raise ValueError('Specify exactly one Bloomberg workbook.')
    PRIVATE_DIR = PROJECT_DIR/'data/private_uploads'
    PRIVATE_DIR.mkdir(parents=True, exist_ok=True)
    BLOOMBERG_XLSX = PRIVATE_DIR/workbooks[0].name
    if workbooks[0] != BLOOMBERG_XLSX:
        shutil.copy2(workbooks[0], BLOOMBERG_XLSX)
    sys.path.insert(0,str(PROJECT_DIR))
    print('Using checked-out source at', PROJECT_DIR)
else:
    if colab_files is not None:
        print('Upload BMLL_4Y_Coverage_Fix_Support.zip AND BMLL_Bloomberg - FINAL.xlsx together.')
        uploaded = colab_files.upload()
        items = [Path(k).resolve() for k in uploaded]
    else:
        print('Local Jupyter: enter paths to the ZIP and Bloomberg XLSX.')
        items = [Path(input('Support ZIP: ').strip()).expanduser().resolve(),
                 Path(input('Bloomberg workbook: ').strip()).expanduser().resolve()]
        if not all(p.is_file() for p in items):
            raise FileNotFoundError('At least one input path does not exist')

    source_zips = []
    for path in items:
        if path.suffix.lower() == '.zip':
            with zipfile.ZipFile(path) as z:
                names = z.namelist()
                if any(x.endswith('scripts/coverage_recovery.py') for x in names) and \
                   any(x.endswith('research_pipeline.py') for x in names):
                    source_zips.append(path)
    if len(source_zips) != 1:
        raise ValueError('Upload exactly one BMLL_4Y_Coverage_Fix_Support.zip (other ZIPs lack coverage recovery).')
    workbooks = [p for p in items if p.suffix.lower() == '.xlsx']
    if len(workbooks) != 1:
        raise ValueError('Upload exactly one Bloomberg .xlsx workbook')

    RUN_ROOT = Path('/content' if colab_files is not None else Path.cwd()) / ('BMLL_4Y_'+uuid.uuid4().hex[:8])
    RUN_ROOT.mkdir(parents=True,exist_ok=False)
    with zipfile.ZipFile(source_zips[0]) as archive:
        for info in archive.infolist():
            p=Path(info.filename)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Untrusted ZIP contains an unsafe path')
        archive.extractall(RUN_ROOT)
    PROJECT_DIR = RUN_ROOT/'BMLL_Notebook_Release_Kit'
    assert (PROJECT_DIR/'scripts/coverage_recovery.py').is_file()
    assert (PROJECT_DIR/'scripts/multiyear_preflight.py').is_file()
    assert (PROJECT_DIR/'scripts/build_wrds_handoff.py').is_file()
    assert (PROJECT_DIR/'scripts/complete_release.py').is_file()
    assert (PROJECT_DIR/'fund_simulator.py').is_file(), 'Support ZIP predates the Friday-auction fix'
    assert (PROJECT_DIR/'BMLL_Stage1_2_Corporate_Actions_WRDS_Bloomberg.ipynb').is_file()
    PRIVATE_DIR = PROJECT_DIR/'data/private_uploads'
    PRIVATE_DIR.mkdir(parents=True,exist_ok=True)
    BLOOMBERG_XLSX = PRIVATE_DIR / workbooks[0].name
    shutil.copy2(workbooks[0],BLOOMBERG_XLSX)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0,str(PROJECT_DIR))
    print('Project:',PROJECT_DIR)
    print('Bloomberg:',BLOOMBERG_XLSX,'| MB:',round(BLOOMBERG_XLSX.stat().st_size/1e6,2))

## 2. Install dependencies and run checks

Requires WRDS Python package, `psycopg2`, Pandas and OpenPyXL. No user credentials are stored in the notebook.

In [ ]:
import subprocess

if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(PROJECT_DIR/'requirements.txt'),
                           'psycopg2-binary>=2.9','openpyxl>=3.1','nbformat>=5'])
if RUN_OFFLINE_TESTS:
    t=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
        cwd=PROJECT_DIR,text=True,capture_output=True)
    print((t.stdout+t.stderr)[-6500:])
    if t.returncode:
        raise RuntimeError('Offline regression tests failed. Do not begin a four-year run.')

## 3. Normalize Bloomberg corporate actions

Converts only the workbook you upload. QA flags and blank PERMNOs are retained for historical WRDS identifier matching, not assigned speculative ticker matches.

In [ ]:
from IPython.display import display
from scripts.extract_bloomberg import convert

BLOOMBERG_CSV = PROJECT_DIR/'data/BMLL_Bloomberg_Corporate_Actions.csv'
BLOOMBERG_QC = PROJECT_DIR/'data/bloomberg_review_flags.csv'
BLOOMBERG_REPORT = PROJECT_DIR/'data/bloomberg_conversion_report.json'
convert(BLOOMBERG_XLSX,BLOOMBERG_CSV,BLOOMBERG_QC,BLOOMBERG_REPORT)
print(json.dumps(json.loads(BLOOMBERG_REPORT.read_text())['counts'],indent=2))
print('Bloomberg export ready; raw workbook remains private.')

## 4. Authenticate WRDS and discover the bootstrap date

The fund's first decision must precede July 2022, because a July 1 initialization would trade with weights calculated after the earlier June rebalance. The date is discovered from your actual CRSP release, not assumed to be a trading day.

With `EXECUTION_TIMING='reference_close'` the replay starts on the **session before the June-2022 reference Friday**, so the initial portfolio is bought in the same closing auction as the index. Index adds then need CRSP closes on the Thursday and Friday before their Monday snapshot. The preflight in section 6 checks this.

In Colab, create `WRDS_USERNAME` and `WRDS_PASSWORD` Secrets and enable notebook access, or use the secure prompts.

In [ ]:
from research_pipeline import load_helpers
helpers = load_helpers(PROJECT_DIR/'BMLL_Stage1_2_Corporate_Actions_WRDS_Bloomberg.ipynb')
if RUN_WRDS_EXTRACTION:
    stock_table = f'{CRSP_SCHEMA}.dsf_v2'
    with helpers['wrds_session']() as db:
        bootstrap_weights, bootstrap_diagnostics = helpers['sp500_cap_weights_quarterly'](
            db, start_date=REBALANCE_START, end_date=ANALYSIS_START,
            membership_schema=CRSP_SCHEMA.replace('_stock','_indexes'),
            stock_schema=CRSP_SCHEMA, on_unavailable='raise')
        prior = bootstrap_weights.loc[bootstrap_weights.date.le(pd.Timestamp(ANALYSIS_START))]
        if prior.empty:
            raise RuntimeError('No WRDS post-rebalance snapshot exists on/before the requested analysis start')
        first = prior.loc[prior.date.eq(prior.date.max())].iloc[0]
        if EXECUTION_TIMING == 'reference_close':
            lo, hi = helpers['get_price_bounds'](db, stock_table)
            # Decision session = last CRSP trading day strictly before the reference Friday.
            REPLAY_START = str(pd.Timestamp(helpers['resolve_effective_date'](
                db, stock_table, pd.Timestamp(first.reference_date) - pd.Timedelta(days=1),
                lo, hi, 7)).date())
        else:
            REPLAY_START = str(pd.Timestamp(first.date).date())
    print('Bootstrap snapshot:', pd.Timestamp(first.date).date(),
          '| reference Friday:', pd.Timestamp(first.reference_date).date())
    print('Replay starts (first decision session):', REPLAY_START)
    print('Analysis starts:', ANALYSIS_START)
else:
    print('Using existing WRDS cache. It must have coverage.json and a prior-quarter snapshot.')
    coverage_file=PROJECT_DIR/'data/stage2_inputs/coverage.json'
    if not coverage_file.is_file():
        raise FileNotFoundError('WRDS extraction is disabled but stage2 cache is absent')
    REPLAY_START=json.loads(coverage_file.read_text())['start']

## 5. Extract four years of CRSP/WRDS data

Pulls quarterly membership/cap-weight snapshots, all relevant daily raw closes, distributions, delistings, adjustment factors and historical identifiers. **Do not proceed if WRDS reports missing coverage.** You can switch to the licensed annual CRSP release in Configuration and rerun in a fresh notebook session.

In [ ]:
from scripts.build_wrds_handoff import run as build_wrds_handoff
if RUN_WRDS_EXTRACTION:
    extraction_summary=build_wrds_handoff(
        REPLAY_START,END_DATE,REBALANCE_START,BLOOMBERG_CSV,stock_schema=CRSP_SCHEMA)
else:
    extraction_summary=json.loads((PROJECT_DIR/'data/stage2_inputs/extraction_summary.json').read_text())
print('WRDS pull summary:')
print(json.dumps(extraction_summary,indent=2))

## 5A. S&P 500 total-return benchmark (WRDS)

Daily `vwretd` from CRSP's S&P 500 index file: the value-weighted return of index constituents **including dividends**, computed from CRSP market caps (the same caps behind `cap_weight_proxy`). It is close to, but not the same as, S&P's float-adjusted SPXT. The table name differs between legacy and CIZ releases and is discovered automatically. A release that ends before `END_DATE` raises an error instead of silently truncating the comparison.

In [ ]:
from scripts.benchmark import fetch_sp500_total_return, save_benchmark, load_benchmark
from research_pipeline import read_table
STAGE_DIR = PROJECT_DIR/'data/stage2_inputs'
if RUN_BENCHMARK_EXTRACTION:
    replay_calendar = read_table(STAGE_DIR/'calendar.csv').date
    with helpers['wrds_session']() as db:
        benchmark = fetch_sp500_total_return(db, REPLAY_START, END_DATE, replay_calendar)
    save_benchmark(benchmark, STAGE_DIR)
else:
    benchmark = load_benchmark(STAGE_DIR)
print('Benchmark:', benchmark.source.iloc[0], '|', benchmark.date.min().date(), '->',
      benchmark.date.max().date(), '|', len(benchmark), 'sessions')
if 'price_return' in benchmark:
    div = (benchmark.benchmark_return - benchmark.price_return).sum() * 252 / len(benchmark)
    print(f'Implied dividend yield (vwretd - vwretx): {div:.2%} p.a.')

## 5B. Synthetic investor flows

A seeded **stress scenario**, not a calibrated investor model. Each dealing session has a normal daily flow (`FLOW_DAILY_SIGMA_BPS` of NAV), plus a rare institutional ticket (probability `FLOW_LUMPY_PROB`, size in `FLOW_LUMPY_BPS`, random sign).

Notice arrives one session ahead. Units are issued or cancelled at the pre-flow NAV on the dealing day. Flow trades fill in that day's closing auction, pro rata to holdings: inflows are invested only once cash exceeds the buffer by `FLOW_BAND_BPS`, and redemptions are always funded. On a rebalance decision day, the next day's flow goes into the rebalance budget.

In [ ]:
from fund_simulator import synthetic_flows
replay_calendar = read_table(STAGE_DIR/'calendar.csv').date
FLOWS = None
if USE_SYNTHETIC_FLOWS:
    FLOWS = synthetic_flows(replay_calendar, seed=FLOW_SEED, daily_sigma_bps=FLOW_DAILY_SIGMA_BPS,
                            lumpy_prob=FLOW_LUMPY_PROB, lumpy_bps=FLOW_LUMPY_BPS)
    FLOWS.to_csv(STAGE_DIR/'synthetic_flows.csv', index=False)
    print(f"{int(FLOWS.is_lumpy.sum())} institutional tickets; daily flow sd "
          f"{FLOWS.flow_bps.std():.1f} bps; cumulative net {FLOWS.flow_bps.sum():+.0f} bps of NAV")
    display(FLOWS.loc[FLOWS.is_lumpy].head(10))
else:
    print('No investor flows (baseline).')

## 6. Mandatory four-year preflight

This corrects an error in the older code: securities should **not** be required to trade on days before joining the fund or after leaving it. Check their active holdings intervals, plus the next execution session after a rebalance. Save quarterly and yearly coverage diagnostics before computing any NAV.

In [ ]:
from scripts.multiyear_preflight import audit
preflight, price_report, quarterly_report = audit(
    PROJECT_DIR,evaluation_start=ANALYSIS_START,evaluation_end=END_DATE,
    execution_timing=EXECUTION_TIMING)
print(json.dumps(preflight,indent=2))
display(quarterly_report)
display(price_report.loc[~price_report.ready].sort_values(
    ['missing_sessions','bad_price_sessions'],ascending=False).head(30))
if EXECUTION_TIMING == 'reference_close' and preflight['securities_with_missing_active_prices']:
    print('NOTE: under Friday-auction timing, index adds need TR closes on the Thursday/Friday '
          'before their Monday snapshot. If build_wrds_handoff only pulls from the snapshot date, '
          'extend its per-security price window by one week.')
print('Review reports: outputs/four_year_preflight/')

## 7. Corrected daily portfolio + order simulation

The scenario reuses historical CRSP raw prices, WRDS/Bloomberg corporate actions, and the same accounting ledger. It (1) applies narrow **provisional** WRDS choices where a future **CRSP-verified split** explains the dividend basis, (2) checks excluded corporate-action dates against each quarterly holding interval, and (3) checks price coverage on that interval instead of blacklisting PERMNOs across four years. Unresolved events in a holding interval still cause an explicit restricted-portfolio exclusion.

Execution uses `fund_simulator.simulate_fund`:
* With `reference_close`, targets are proportional to CRSP index shares, sized from the decision-day NAV at decision-day prices, and filled at the reference Friday's CRSP close. That close is a proxy for the closing-auction price; BMLL auction prints replace it in the TCA notebook.
* Orders now carry `order_id`, `reason` (INITIAL / REBALANCE / INFLOW / OUTFLOW) and the expected execution date.

The notebook **never auto-approves** the actual `event_decisions.csv`. If the pipeline reports a blocked certification status, treat the outputs as research data only.

In [ ]:
from scripts.complete_release import generate
from traceback import format_exc
CANDIDATE = None
REPLAY_EXCEPTION = None

if RUN_REPLAY:
    try:
        CANDIDATE = generate(
            PROJECT_DIR, REPLAY_START, END_DATE,
            initial_cash=AUM,cash_buffer_bps=CASH_BUFFER_BPS,fee_bps=FEE_BPS,
            asof=str(datetime.now(timezone.utc).date()),mode='restricted',
            workbook=BLOOMBERG_XLSX, qc=BLOOMBERG_QC,
            provisional_wrds_research=PROVISIONAL_WRDS_RESEARCH,
            execution_timing=EXECUTION_TIMING, flows=FLOWS, flow_band_bps=FLOW_BAND_BPS)
        print('Four-year run completed. Release status:',CANDIDATE['report']['status'])
        print('Human review:',json.dumps(CANDIDATE['review_summary'],indent=2))
    except Exception as exc:
        REPLAY_EXCEPTION=f'{type(exc).__name__}: {exc}'
        (PROJECT_DIR/'outputs').mkdir(exist_ok=True)
        (PROJECT_DIR/'outputs/replay_error.txt').write_text(format_exc())
        print('Replay did NOT complete. Full portfolio/orders are NOT available.')
        print(REPLAY_EXCEPTION)
else:
    print('Replay skipped. Only extraction/preflight outputs are available.')

## 8. Verify that order book, portfolio and freeze checks are not being overstated

If a candidate exists, review its status and restricted-portfolio exclusions. Results dated before July 1, 2022 are bootstrap-only and should not be counted in the requested four-year performance evaluation. Do **not** trim accounting source files before validation.

In [ ]:
import numpy as np
FINAL_FOLDER = Path(CANDIDATE['candidate_dir']) if CANDIDATE else None
nav = book = None
if FINAL_FOLDER:
    report=json.loads((FINAL_FOLDER/'validation_report.json').read_text())
    nav=pd.read_csv(FINAL_FOLDER/'portfolio_nav.csv',parse_dates=['date'])
    book=pd.read_csv(FINAL_FOLDER/'orderbook.csv',parse_dates=['decision_date'])
    pos=pd.read_csv(FINAL_FOLDER/'portfolio_positions.csv',parse_dates=['date'])
    excluded_snapshots=pd.read_csv(FINAL_FOLDER/'excluded_security_snapshots.csv')
    analysis_nav=nav[nav.date.ge(ANALYSIS_START)]
    print('Release:',report['status'],'| checks_passed:',report['checks_passed'])
    print('Trading sessions in evaluation period:',len(analysis_nav))
    print('Position rows in evaluation period:',int(pos.date.ge(ANALYSIS_START).sum()))
    print('Orders in evaluation period:',int(book.decision_date.ge(ANALYSIS_START).sum()))
    print('Excluded snapshot rows:',len(excluded_snapshots))
    print('Validation issues:')
    display(pd.DataFrame(report.get('issues',[])).head(40))
    # Rebalance timing check: every REBALANCE fill must land on its reference Friday.
    sched = pd.read_csv(PROJECT_DIR/'outputs/daily_fund/rebalance_schedule.csv',
                        parse_dates=['snapshot_date','reference_date','decision_date','execution_date'])
    sched['execution_weekday'] = sched.execution_date.dt.day_name()
    display(sched)
    if EXECUTION_TIMING == 'reference_close' and not sched.execution_date.eq(sched.reference_date).all():
        raise AssertionError('A rebalance did not execute on its reference Friday')
    # Base = last NAV *before* the window, so the first evaluation day's return counts.
    before = nav[nav.date.lt(ANALYSIS_START)]
    if len(analysis_nav) and len(before):
        first = before.iloc[-1].unit_nav; last = analysis_nav.iloc[-1].unit_nav
        print('Evaluation-period unit NAV return:',f'{last/first-1:.3%}',
              '(restricted proxy; see 8B for the benchmark comparison)')
else:
    print('No replay output was produced; inspect price coverage and replay_error.txt. No dataset is frozen.')

## 8A. Verify recovered target weights per quarter

This is a **measured output from the newly rerun portfolio**, not a projection based on the old ZIP. It compares original WRDS cap-weight targets with the updated scenario's excluded weights and lists the highest remaining exclusions. If the live extraction or replay did not complete, this section will not claim any improvement.

In [ ]:
if CANDIDATE is not None:
    origin = pd.read_csv(PROJECT_DIR/'data/stage2_inputs/weights.csv')
    excluded_snapshots = pd.read_csv(FINAL_FOLDER/'excluded_security_snapshots.csv')
    all_dates = sorted(origin.date.unique())
    loss = excluded_snapshots.groupby('date').cap_weight_proxy.sum().reindex(all_dates,fill_value=0.)
    cover = pd.DataFrame({'decision_date':all_dates,
                          'excluded_target_pct':loss.values*100,
                          'retained_target_pct':(1-loss.values)*100})
    audit_folder=PROJECT_DIR/'outputs/coverage_recovery'
    audit_folder.mkdir(parents=True,exist_ok=True)
    cover.to_csv(audit_folder/'quarterly_coverage_after_live_replay.csv',index=False)
    print('EXCLUSIONS AFTER THE ACTUAL WRDS REPLAY:')
    display(cover)
    print('Range excluded:', f"{cover.excluded_target_pct.min():.2f}% – {cover.excluded_target_pct.max():.2f}%")
    print('Still excluded security-period rows:',len(excluded_snapshots),'distinct PERMNO:',excluded_snapshots.permno.nunique())
    print('Provisional source choices:',CANDIDATE['review_summary'].get('provisional_wrds_disagreements'))
    print('Reviewed human event approvals:',CANDIDATE['review_summary'].get('human_approved_events'))
    if cover.excluded_target_pct.max()>5:
        print('WARNING: More than 5% target weight still excluded in at least one quarter')
    if not CANDIDATE['report'].get('checks_passed',False):
        print('CERTIFICATION BLOCKED: research simulation only. Outstanding items are in review/ CSVs.')
else:
    print('No completed live replay. Do not infer corrected exclusion weights from the old ZIP.')

## 8B. Fund vs S&P 500 total return, tracking error and flows

Unit-NAV returns over the evaluation window (bootstrap sessions excluded), so investor flows do not count as performance. Tracking difference measures drag: cash buffer, whole-share rounding and the excluded restricted securities. Tracking error measures volatility. Under Friday-auction timing, rebalance-day spikes in active return should largely disappear compared with the `next_session` run.

In [ ]:
from scripts.benchmark import performance_report
import matplotlib.pyplot as plt
if nav is not None:
    perf_daily, perf, perf_years = performance_report(nav, benchmark, ANALYSIS_START, END_DATE)
    print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in perf.items()}, indent=2))
    display(perf_years.style.format({'fund_return':'{:.2%}','benchmark_return':'{:.2%}',
        'te_bps':'{:.1f}','tracking_difference_bps':'{:+.1f}','net_flows':'${:,.0f}'}))
    out = PROJECT_DIR/'outputs/benchmark'; out.mkdir(parents=True, exist_ok=True)
    perf_daily.to_csv(out/'fund_vs_sp500tr_daily.csv', index=False)
    perf_years.to_csv(out/'fund_vs_sp500tr_by_year.csv')
    (out/'fund_vs_sp500tr_summary.json').write_text(json.dumps(perf, indent=2))
    sched = pd.read_csv(PROJECT_DIR/'outputs/daily_fund/rebalance_schedule.csv', parse_dates=['execution_date'])
    on_reb = perf_daily.date.isin(sched.execution_date)
    print(f"Mean |active| on rebalance execution days: {1e4*perf_daily.active_return[on_reb].abs().mean():.2f} bps "
          f"vs other days {1e4*perf_daily.active_return[~on_reb].abs().mean():.2f} bps")
    fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax[0].plot(perf_daily.date, 1e4*perf_daily.cumulative_active, color='#2a5c8a')
    ax[0].set_ylabel('Cumulative active (bps)'); ax[0].axhline(0, color='grey', lw=.6)
    ax[1].plot(perf_daily.date, perf_daily.rolling_63d_te_bps, color='#2a5c8a')
    ax[1].set_ylabel('Rolling 63d TE (bps)')
    for d in sched.execution_date:
        for a in ax: a.axvline(d, color='#bbbbbb', lw=.5, zorder=0)
    ax[0].set_title('Fund vs S&P 500 total return (grey lines: rebalance auctions)')
    fig.tight_layout(); plt.show()
    flows_nav = nav.loc[nav.external_flow.ne(0)]
    if len(flows_nav):
        print(f"Investor flows: {len(flows_nav)} dealing days, gross ${flows_nav.external_flow.abs().sum():,.0f}, "
              f"net ${flows_nav.external_flow.sum():,.0f}")
        if 'reason' in book:
            display(book.groupby('reason').agg(orders=('permno','size')))
else:
    print('No replay output: benchmark comparison skipped.')

## 9. Prioritize the four-year corporate-action review

Ranks unresolved events by latest observed portfolio weights and event complexity. These are **review priorities only**, not automatically approved actions.

In [ ]:
from scripts.review_priority_4year import prioritize
if CANDIDATE is not None:
    queue=prioritize(PROJECT_DIR)
    pending=queue.loc[queue.decision.fillna('PENDING').eq('PENDING')]
    print('Pending/blocked event candidates:',len(pending))
    display(pending[['review_attention','permno','event_type','ex_date','match_status',
        'last_observed_target_weight','review_key']].head(40))
    print('Saved prioritized_event_decisions_4y.csv and prioritized_scope_decisions_4y.csv')
else:
    print('Review queue cannot be ranked until a candidate has created event_decisions.csv')

## 10. Export CSVs, audit and review queues

Creates one downloadable ZIP. Licensed market-data tables should remain restricted to authorized project members. **Do not upload the workbook or WRDS raw files to public GitHub.**

Even when arithmetic passes, unresolved actions/excluded securities are explicitly marked **RESEARCH_CANDIDATE_NOT_CERTIFIED**.

In [ ]:
import zipfile
from scripts.complete_release import digest
OUT_ZIP=RUN_ROOT/f'BMLL_4Y_{ANALYSIS_START}_{END_DATE}_COVERAGE_CORRECTED_RESEARCH.zip'
folders=[
 (PROJECT_DIR/'outputs/four_year_preflight','preflight'),
 (PROJECT_DIR/'data/stage2_review','review'),
 (PROJECT_DIR/'outputs/coverage_recovery','coverage_recovery'),
 (PROJECT_DIR/'outputs/benchmark','benchmark'),
]
if FINAL_FOLDER is not None:
    folders.insert(0,(FINAL_FOLDER,'candidate'))
with zipfile.ZipFile(OUT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for folder,arc_prefix in folders:
        if folder.is_dir():
            for p in sorted(folder.glob('*')):
                if p.is_file() and p.suffix.lower() in ('.csv','.json','.txt'):
                    z.write(p,arcname=f'{arc_prefix}/{p.name}')
    p=PROJECT_DIR/'outputs/daily_fund/rebalance_schedule.csv'
    if p.exists():z.write(p,arcname='candidate/rebalance_schedule.csv')
    for filename in ['extraction_summary.json','coverage.json','benchmark_sp500_tr.csv',
                     'benchmark_sp500_tr.json','synthetic_flows.csv']:
        p=PROJECT_DIR/'data/stage2_inputs'/filename
        if p.exists():z.write(p,arcname=f'preflight/{filename}')
    p=PROJECT_DIR/'outputs/replay_error.txt'
    if p.exists():z.write(p,arcname='preflight/replay_error.txt')
print('Export:',OUT_ZIP,'size MB:',round(OUT_ZIP.stat().st_size/1e6,2))
if colab_files is not None:colab_files.download(str(OUT_ZIP))
else:print('Local file:',OUT_ZIP)

## Completion criteria and what blocks a true freeze

A **four-year full-index release** additionally requires: reviewed corporate actions and scope records, no unexplained constituent exclusions or active-date price gaps, and correct handling of mergers/spin-offs/delistings/fractional shares. The **daily index total-return benchmark** (section 5A) and **closing-auction rebalance timing** (section 7) are now implemented. Dated BMLL identifiers are still needed for the execution handoff.

The current simulator has no L2/L3 limit-order-book reconstruction. `orderbook.csv` is the **portfolio manager's order blotter**. Its fills use the CRSP close of the reference Friday as a closing-auction proxy, not executed auction prints. Closing auction vs VWAP vs arrival-plus-impact execution is compared in `BMLL_TCA_Execution_Comparison.ipynb`. Economic exceptions are surfaced as review CSVs; the code will never invent treatments to force certification.